# LLM Science Feature Blend

ExtraTrees option classifier blended with length + TF-IDF handcrafted score.


In [ ]:
import csv
import math
import re
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.ensemble import ExtraTreesClassifier

OPTIONS = ("A", "B", "C", "D", "E")
TFIDF_WEIGHT = 0.25
CLASSIFIER_WEIGHT = 2.0
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
STOPWORDS = {
    "the", "of", "and", "to", "in", "a", "is", "are", "which", "following",
    "statement", "statements", "accurately", "describes", "describe", "what", "how",
    "by", "for", "with", "as", "an", "on", "from", "that", "it", "this", "its",
    "be", "was", "were", "can", "into", "or", "not", "due", "between", "among",
    "about", "refers", "system", "systems", "object", "objects", "law", "theory",
    "mass", "data", "time", "answer", "option",
}


def find_csv(name):
    candidates = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    if not candidates:
        return None
    return candidates[0]


def read_rows(path):
    if path is None:
        return []
    with path.open(newline="", encoding="utf-8") as src:
        return list(csv.DictReader(src))


def tokens(text):
    return [word.lower() for word in WORD_RE.findall(text or "")]


def content_tokens(text):
    return [word for word in tokens(text) if len(word) >= 3 and word not in STOPWORDS]


def word_count(text):
    return len(tokens(text))


def zscores(values):
    values = np.asarray(values, dtype=float)
    scale = values.std() or 1.0
    return (values - values.mean()) / scale


def build_idf(rows):
    docs = []
    for row in rows:
        docs.append(set(content_tokens(row["prompt"])))
        for option in OPTIONS:
            docs.append(set(content_tokens(row[option])))
    df = Counter()
    for doc in docs:
        df.update(doc)
    n_docs = len(docs)
    return {term: math.log((n_docs + 1) / (count + 1)) + 1 for term, count in df.items()}


def tfidf_vector(words, idf):
    counts = Counter(words)
    return {word: count * idf.get(word, 1.0) for word, count in counts.items()}


def cosine(a, b):
    numerator = sum(value * b.get(key, 0.0) for key, value in a.items())
    norm_a = math.sqrt(sum(value * value for value in a.values()))
    norm_b = math.sqrt(sum(value * value for value in b.values()))
    if not norm_a or not norm_b:
        return 0.0
    return numerator / (norm_a * norm_b)


def row_features(row, idf, with_label=False):
    prompt_tokens = content_tokens(row["prompt"])
    prompt_set = set(prompt_tokens)
    prompt_vec = tfidf_vector(prompt_tokens, idf)
    prompt_len = word_count(row["prompt"])
    prompt_content_len = len(prompt_tokens)

    lengths = [word_count(row[option]) for option in OPTIONS]
    tfidf_values = [
        cosine(prompt_vec, tfidf_vector(content_tokens(row[option]), idf))
        for option in OPTIONS
    ]
    overlaps = []
    for option in OPTIONS:
        option_set = set(content_tokens(row[option]))
        overlaps.append(len(prompt_set & option_set) / (len(prompt_set | option_set) or 1))

    length_z = zscores(lengths)
    tfidf_z = zscores(tfidf_values)
    overlap_z = zscores(overlaps)
    hand_scores = length_z + TFIDF_WEIGHT * tfidf_z

    features = []
    labels = []
    for idx, option in enumerate(OPTIONS):
        features.append([
            lengths[idx],
            length_z[idx],
            tfidf_values[idx],
            tfidf_z[idx],
            overlaps[idx],
            overlap_z[idx],
            idx,
            prompt_len,
            prompt_content_len,
        ])
        if with_label:
            labels.append(1 if row["answer"] == option else 0)
    return features, labels, hand_scores


train_path = find_csv("train.csv")
test_path = find_csv("test.csv")
if train_path is None or test_path is None:
    raise FileNotFoundError("Both train.csv and test.csv are required under /kaggle/input")

train_rows = read_rows(train_path)
test_rows = read_rows(test_path)
idf = build_idf(train_rows + test_rows)

X_train = []
y_train = []
for row in train_rows:
    features, labels, _ = row_features(row, idf, with_label=True)
    X_train.extend(features)
    y_train.extend(labels)

model = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=3,
    min_samples_leaf=8,
    random_state=2,
    class_weight="balanced",
)
model.fit(np.asarray(X_train, dtype=float), np.asarray(y_train, dtype=int))

output_path = Path("/kaggle/working/submission.csv")
with output_path.open("w", newline="", encoding="utf-8") as dst:
    writer = csv.DictWriter(dst, fieldnames=["id", "prediction"])
    writer.writeheader()
    for row in test_rows:
        features, _, hand_scores = row_features(row, idf, with_label=False)
        clf_scores = model.predict_proba(np.asarray(features, dtype=float))[:, 1]
        final_scores = hand_scores + CLASSIFIER_WEIGHT * zscores(clf_scores)
        ranked = [
            option
            for _, option in sorted(zip(final_scores, OPTIONS), key=lambda item: (-item[0], item[1]))
        ]
        writer.writerow({"id": row["id"], "prediction": " ".join(ranked[:3])})

print(f"Using train: {train_path}")
print(f"Using test: {test_path}")
print(f"Rows: train={len(train_rows)} test={len(test_rows)}")
print(f"Wrote {output_path}")
